### Setting up Training Data ###

In [ ]:
from __future__ import print_function
import numpy as np
import h5py
import csv
import matplotlib.pyplot as plt
import torch
from torch.nn import Linear, ReLU, CrossEntropyLoss, Sequential, Conv2d, MaxPool2d, Module, Softmax, BatchNorm2d, Dropout
from torch.optim import Adam, SGD
from torch.autograd import Variable
import torch.nn.functional as F
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import torchvision
from torchvision import datasets, transforms

with h5py.File("data.hdf5", 'r') as f:
    train_data = f['train'][()]
    test_data = f['test'][()]
    
labels = open('train_labels.csv')
labels_reader = csv.reader(labels)
classes = ["FM","OQPSK","BPSK","8PSK","AM-SSB-SC","4ASK","16PSK","AM-DSB-SC","QPSK","OOK"]
train_labels_data = []

for row in labels_reader:
    if row[0] != 'Id':
        train_labels_data.append(classes.index(row[1]))

train_labels_data = np.asarray(train_labels_data)

print(f"Train data shape: {train_data.shape}")
print(f"Train labels length: {len(train_labels_data)}")
print(f"Test data shape: {test_data.shape}")

### Data Augmentation and Training/Validation Set Processing ###

In [ ]:
# FIXED: Proper data augmentation by creating lists and concatenating
print("Applying data augmentation...")

augmented_data = [train_data]  # Start with original data
augmented_labels = [train_labels_data]  # Start with original labels

# Apply 90-degree rotation augmentation (first 4000 samples only to manage memory)
for idx, sample in enumerate(train_data[:4000]):
    if idx % 1000 == 0:
        print(f"Augmenting sample {idx}")
    
    # 90-degree rotation in IQ plane
    theta = np.pi/2
    rot_90 = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    rotated_90 = np.dot(rot_90, sample.transpose()).transpose()
    
    # 180-degree rotation in IQ plane
    theta = np.pi
    rot_180 = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    rotated_180 = np.dot(rot_180, sample.transpose()).transpose()
    
    # Add rotated samples and their labels
    augmented_data.extend([rotated_90, rotated_180])
    augmented_labels.extend([train_labels_data[idx], train_labels_data[idx]])

# Convert back to numpy arrays
train_data_aug = np.array(augmented_data)
train_labels_aug = np.array(augmented_labels)

print(f"Augmented data shape: {train_data_aug.shape}")
print(f"Augmented labels shape: {train_labels_aug.shape}")

In [ ]:
# Converting training data into torch format
train_x = train_data_aug.reshape(-1, 1, 1024, 2)  # Use augmented data
train_x = torch.from_numpy(train_x).float()

# Converting test data into torch format
test_x = test_data.reshape(20000, 1, 1024, 2)
test_x = torch.from_numpy(test_x).float()

# Converting the target into torch format
train_y = train_labels_aug.astype(int)
train_y = torch.from_numpy(train_y).long()

# FIXED: Better train/val split with proper validation size
train_x, val_x, train_y, val_y = train_test_split(
    train_x, train_y, test_size=0.2, shuffle=True, random_state=42, stratify=train_y
)

# Create data loaders
train_dataset = torch.utils.data.TensorDataset(train_x, train_y)
val_dataset = torch.utils.data.TensorDataset(val_x, val_y)

trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
valloader = torch.utils.data.DataLoader(val_dataset, batch_size=64, shuffle=False)

print(f"Training set: {train_x.shape}")
print(f"Validation set: {val_x.shape}")
print(f"Training labels: {train_y.shape}")
print(f"Validation labels: {val_y.shape}")

### Visualizing Data ###

In [ ]:
# Visualize a few samples from each class
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for class_idx in range(len(classes)):
    # Find first sample of this class
    sample_idx = np.where(train_labels_data == class_idx)[0][0]
    sample = train_data[sample_idx]
    
    I = sample[:, 0]  # In-phase
    Q = sample[:, 1]  # Quadrature
    
    axes[class_idx].scatter(I, Q, alpha=0.6, s=1)
    axes[class_idx].set_title(f'{classes[class_idx]}')
    axes[class_idx].set_xlabel('I')
    axes[class_idx].set_ylabel('Q')
    axes[class_idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### FIXED: Improved CNN Architecture ###

In [ ]:
import torch.nn as nn

class ImprovedNet(nn.Module):
    def __init__(self, num_classes=10):
        super(ImprovedNet, self).__init__()
        
        # FIXED: Separate batch norm layers for each conv layer
        self.conv1 = nn.Conv2d(1, 256, kernel_size=(3,1), stride=1, padding=(1,0))
        self.bn1 = nn.BatchNorm2d(256)
        
        self.conv2 = nn.Conv2d(256, 256, kernel_size=(3,2), stride=1, padding=(1,0))
        self.bn2 = nn.BatchNorm2d(256)
        
        self.conv3 = nn.Conv2d(256, 128, kernel_size=(3,1), stride=1, padding=(1,0))
        self.bn3 = nn.BatchNorm2d(128)
        
        self.conv4 = nn.Conv2d(128, 80, kernel_size=(3,1), stride=1, padding=(1,0))
        self.bn4 = nn.BatchNorm2d(80)
        
        # Global average pooling to reduce parameters
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        self.dropout = nn.Dropout(p=0.5)
        self.fc1 = nn.Linear(80, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Conv layers with proper batch norm
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        
        # Global average pooling instead of flattening large feature maps
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        
        # Fully connected layers
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        
        return x  # Return raw logits, not log_softmax

# Test the network architecture
def test_network_shape():
    net = ImprovedNet(len(classes))
    sample_input = torch.randn(4, 1, 1024, 2)
    output = net(sample_input)
    print(f"Network output shape: {output.shape}")
    return net

cnn = test_network_shape()
print(cnn)
print(f"\nTotal parameters: {sum(p.numel() for p in cnn.parameters()):,}")

### FIXED: Improved Training Protocol ###

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from tqdm import tqdm

def train_model(model, train_loader, val_loader, num_epochs=50, device='cpu'):
    """FIXED: Improved training with proper validation, early stopping, and LR scheduling"""
    
    model = model.to(device)
    
    # FIXED: Better optimizer and loss function
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)
    criterion = nn.CrossEntropyLoss()
    
    # Early stopping parameters
    best_val_loss = float('inf')
    patience_counter = 0
    patience = 10
    
    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
        for batch_idx, (data, target) in enumerate(progress_bar):
            data, target = data.to(device), target.to(device)
            
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            pred = output.argmax(dim=1)
            train_correct += pred.eq(target).sum().item()
            train_total += target.size(0)
            
            # Update progress bar
            progress_bar.set_postfix({
                'Loss': f'{loss.item():.4f}',
                'Acc': f'{100.*train_correct/train_total:.2f}%'
            })
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = criterion(output, target)
                
                val_loss += loss.item()
                pred = output.argmax(dim=1)
                val_correct += pred.eq(target).sum().item()
                val_total += target.size(0)
        
        # Calculate averages
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        train_acc = 100. * train_correct / train_total
        val_acc = 100. * val_correct / val_total
        
        # Store metrics
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        train_accuracies.append(train_acc)
        val_accuracies.append(val_acc)
        
        # Update learning rate
        scheduler.step(avg_val_loss)
        
        print(f'Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.2f}%, '
              f'Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.2f}%')
        
        # Early stopping and model saving
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': avg_val_loss,
                'val_acc': val_acc
            }, 'best_model.pt')
            print(f'New best model saved with validation loss: {best_val_loss:.4f}')
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            print(f'Early stopping after {patience} epochs without improvement')
            break
    
    return train_losses, val_losses, train_accuracies, val_accuracies

# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Create fresh model instance
model = ImprovedNet(len(classes))

In [ ]:
# Train the model
train_losses, val_losses, train_accs, val_accs = train_model(
    model, trainloader, valloader, num_epochs=50, device=device
)

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, label='Training Loss')
ax1.plot(val_losses, label='Validation Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(train_accs, label='Training Accuracy')
ax2.plot(val_accs, label='Validation Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

### FIXED: Proper Predictions and Submission ###

In [ ]:
import pandas as pd

# Load best model
checkpoint = torch.load('best_model.pt', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded model with validation accuracy: {checkpoint['val_acc']:.2f}%")

# Generate predictions
model.eval()
model = model.to(device)
test_x = test_x.to(device)

predictions = []
batch_size = 64

with torch.no_grad():
    for i in tqdm(range(0, len(test_x), batch_size), desc='Generating predictions'):
        batch = test_x[i:i+batch_size]
        output = model(batch)
        pred = output.argmax(dim=1).cpu().numpy()
        predictions.extend(pred)

# Convert predictions to class names
pred_labels = [classes[pred] for pred in predictions]

print(f"Generated {len(pred_labels)} predictions")
print(f"Prediction distribution:")
from collections import Counter
pred_counts = Counter(pred_labels)
for class_name, count in pred_counts.items():
    print(f"  {class_name}: {count} ({100*count/len(pred_labels):.1f}%)")

In [ ]:
# FIXED: Proper submission format without extra index column
submission_df = pd.DataFrame({
    'Id': range(len(pred_labels)),
    'Category': pred_labels
})

# Save without index to avoid extra column
submission_df.to_csv('submission_fixed.csv', index=False)

print("Submission file saved as 'submission_fixed.csv'")
print("\nFirst 10 rows of submission:")
print(submission_df.head(10))

# Also save predictions as numpy array for later use
np.save('predictions_fixed.npy', predictions)
print("\nPredictions saved as 'predictions_fixed.npy'")